[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/37.%20The%20Equipment%20Selection%20Problem%20%28RTG%20vs.%20RMG%20vs.%20Straddle%29/P37-Tier-1_executed.ipynb)

---



# 37. The Equipment Selection Problem (RTG vs. RMG vs. Straddle)
## Tier 1: The Pen-and-Paper Method — Multi-Objective Mathematical Optimization

### Goal and workflow

This notebook compares integer equipment configurations for a container terminal. It first enumerates every configuration that satisfies the model's constraints, then identifies the Pareto-optimal configurations, and finally applies a weighted-sum score under several preference scenarios. The same enumeration is used for the budget sensitivity analysis.

The inputs are the equipment data below, a capital budget, and a minimum productivity requirement. The outputs are feasible equipment counts, their modeled performance measures, the Pareto set, and the highest-scoring feasible configuration for each set of weights. The method is exhaustive for the stated model and bounds; it is not a mixed-integer programming solver.

### Model assumptions and scope

- Equipment counts are nonnegative integers, and each unit contributes additively to capital cost and productivity.
- Flexibility is the unweighted-by-capacity average of the flexibility scores of all selected units. Sustainability is a score based on average energy consumption per unit. These are model proxies, not complete operational or environmental assessments.
- The budget applies to capital cost only. Although operating cost and maintenance data are included in the equipment records, neither is part of the objective or constraints in this model. There is no planning horizon or discounting.
- A feasible configuration must meet the capital budget, the minimum productivity requirement, and the minimum equipment count of three units.
- The model does not represent uncertainty, equipment interactions, land or footprint limits, staffing, service reliability, emissions, or detailed terminal operations. Results are conditional on the stated data and assumptions.

The concrete example uses a capital budget of $50 million and a minimum productivity requirement of 500 containers per hour. The equipment types are RTG, RMG, and Straddle, with the data defined in the next section.

### Mathematical model

Let $I = \{\mathrm{RTG}, \mathrm{RMG}, \mathrm{Straddle}\}$ be the equipment types, and let $x_i \in \mathbb{Z}_{\geq 0}$ be the number of units of type $i$. For each type, let $c_i$ be capital cost in millions of dollars, $p_i$ be productivity in containers per hour, $f_i$ be flexibility, and $e_i$ be energy consumption in kWh per container as recorded in the data. The model uses the following measures:

$$
C(x) = \sum_{i \in I} c_i x_i, \qquad
P(x) = \sum_{i \in I} p_i x_i, \qquad
F(x) = \frac{\sum_{i \in I} f_i x_i}{\sum_{i \in I} x_i}, \qquad
E(x) = \frac{\sum_{i \in I} e_i x_i}{\sum_{i \in I} x_i}, \qquad
S(x) = \frac{1}{1 + E(x)}.
$$

Here, $C(x)$ is capital cost, $P(x)$ is total modeled productivity, $F(x)$ is average flexibility, $E(x)$ is average energy consumption per unit, and $S(x)$ is the sustainability proxy. The formulas for $F(x)$, $E(x)$, and $S(x)$ require at least one unit; the feasibility rule below requires at least three.

The feasible set is

$$
\mathcal{X} = \left\{x \in \mathbb{Z}_{\geq 0}^{|I|} :
C(x) \leq B,\quad P(x) \geq P_{\min},\quad
\sum_{i \in I} x_i \geq 3 \right\},
$$

where $B$ is the capital budget and $P_{\min}$ is the minimum required productivity. The objectives are to minimize $C(x)$ and maximize $P(x)$, $F(x)$, and $S(x)$. These objectives can conflict, so the model does not imply a single best configuration without additional preferences.

A feasible solution $x$ is Pareto-dominated if another feasible solution is no more costly, no less productive, no less flexible, and no less sustainable, with a strict improvement in at least one of these measures. The Pareto set contains the feasible solutions that are not dominated. The weighted-sum step assigns a preference score to each feasible configuration; it selects a preferred configuration for the chosen weights, but it is not a guarantee that every Pareto solution can be generated by some weight vector.

The enumeration examines each equipment count from zero up to the largest count of that type that could individually fit within the budget. Its search size is the product of the three count-range sizes. If there are $F$ feasible configurations, the direct pairwise Pareto comparison takes $O(F^2)$ comparisons. This exhaustive approach is practical for the small, bounded example and can become expensive as the number of equipment types or count ranges grows.

In [ ]:
from dataclasses import dataclass
import math
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

np.random.seed(42)

print('Equipment Selection — Multi-Objective Mathematical Optimization')
print('Exhaustive enumeration, weighted sums, and Pareto analysis')

### Equipment data structure

Each equipment record stores the attributes used by the model as well as operating cost, maintenance, and footprint data. Keeping these fields together makes the input data explicit; in this model, only capital cost, productivity, flexibility, and energy consumption enter the calculations.

In [ ]:
@dataclass
class EquipmentType:
    name: str
    capital_cost: float  # Million USD
    operating_cost: float  # Million USD/year
    productivity: float  # Containers/hour
    flexibility: float  # Scale from 0 to 1
    maintenance: float  # Million USD/year
    footprint: float  # Square meters
    energy_consumption: float  # kWh/container


equipment_types = {
    'RTG': EquipmentType(
        name='Rubber-Tired Gantry',
        capital_cost=2.5,
        operating_cost=0.8,
        productivity=25,
        flexibility=0.8,
        maintenance=0.3,
        footprint=200,
        energy_consumption=3.2,
    ),
    'RMG': EquipmentType(
        name='Rail-Mounted Gantry',
        capital_cost=3.2,
        operating_cost=0.6,
        productivity=35,
        flexibility=0.4,
        maintenance=0.25,
        footprint=150,
        energy_consumption=2.8,
    ),
    'Straddle': EquipmentType(
        name='Straddle Carrier',
        capital_cost=4.1,
        operating_cost=1.1,
        productivity=20,
        flexibility=0.9,
        maintenance=0.4,
        footprint=100,
        energy_consumption=4.5,
    ),
}

print('Equipment types defined:')
for equipment_code, equipment in equipment_types.items():
    print(f'{equipment_code}: {equipment.name}')
    print(f'  Capital cost: ${equipment.capital_cost}M')
    print(f'  Operating cost: ${equipment.operating_cost}M/year')
    print(f'  Productivity: {equipment.productivity} containers/hour')
    print(f'  Flexibility: {equipment.flexibility:.1f}')
    print()

### Multi-objective optimization implementation

A configuration is represented by three integer counts. `evaluate_solution` computes its model measures and checks feasibility. `generate_all_solutions` performs the exhaustive search, `find_pareto_optimal` removes dominated feasible configurations, and `weighted_sum_optimization` applies one preference vector to the same feasible set.

For a weight vector $w$, the score used here is

$$
Q_w(x) = w_C\left(1 - \frac{C(x)}{B}\right)
+ w_P\frac{P(x)}{1000}
+ w_F F(x) + w_S S(x),
$$

where the weights are nonnegative and sum to one. Larger scores are preferred. The cost term rewards lower capital cost; the other terms reward higher productivity, flexibility, and sustainability. The productivity scale factor is a chosen scoring convention, not a unit conversion, and the resulting terms are not guaranteed to have identical ranges. A weighted score therefore expresses the specified preferences rather than an intrinsic measure of quality. Tied maximum scores may correspond to multiple configurations; the implementation returns the first maximum encountered in enumeration order.

In [ ]:
class MultiObjectiveEquipmentOptimizer:
    def __init__(
        self,
        equipment_types: Dict[str, EquipmentType],
        budget_limit: float = 50.0,
        min_productivity: float = 500,
    ):
        if budget_limit <= 0:
            raise ValueError('The budget limit must be positive.')
        required_types = {'RTG', 'RMG', 'Straddle'}
        if not required_types.issubset(equipment_types):
            raise ValueError('Equipment data must include RTG, RMG, and Straddle.')

        self.equipment_types = equipment_types
        self.budget_limit = budget_limit
        self.min_productivity = min_productivity

        # Each type's individual budget bound keeps enumeration complete while finite.
        self.max_rtg = int(budget_limit / equipment_types['RTG'].capital_cost)
        self.max_rmg = int(budget_limit / equipment_types['RMG'].capital_cost)
        self.max_straddle = int(
            budget_limit / equipment_types['Straddle'].capital_cost
        )

        self.solutions: List[Dict] = []
        self.pareto_solutions: List[Dict] = []

    def evaluate_solution(self, rtg: int, rmg: int, straddle: int) -> Dict:
        """Calculate modeled objectives and test a configuration's feasibility."""
        total_cost = (
            rtg * self.equipment_types['RTG'].capital_cost
            + rmg * self.equipment_types['RMG'].capital_cost
            + straddle * self.equipment_types['Straddle'].capital_cost
        )
        total_productivity = (
            rtg * self.equipment_types['RTG'].productivity
            + rmg * self.equipment_types['RMG'].productivity
            + straddle * self.equipment_types['Straddle'].productivity
        )
        total_units = rtg + rmg + straddle

        if total_units == 0:
            return {
                'feasible': False,
                'cost': total_cost,
                'productivity': total_productivity,
                'flexibility': 0.0,
                'sustainability': 0.0,
                'total_units': 0,
                'avg_energy': 0.0,
            }

        weighted_flexibility = (
            self.equipment_types['RTG'].flexibility * rtg
            + self.equipment_types['RMG'].flexibility * rmg
            + self.equipment_types['Straddle'].flexibility * straddle
        ) / total_units

        total_energy = (
            self.equipment_types['RTG'].energy_consumption * rtg
            + self.equipment_types['RMG'].energy_consumption * rmg
            + self.equipment_types['Straddle'].energy_consumption * straddle
        )
        avg_energy = total_energy / total_units
        sustainability_score = 1.0 / (1.0 + avg_energy)

        feasible = (
            total_cost <= self.budget_limit
            and total_productivity >= self.min_productivity
            and total_units >= 3
        )

        return {
            'feasible': feasible,
            'cost': total_cost,
            'productivity': total_productivity,
            'flexibility': weighted_flexibility,
            'sustainability': sustainability_score,
            'total_units': total_units,
            'avg_energy': avg_energy,
        }

    def generate_all_solutions(self) -> List[Dict]:
        """Enumerate every feasible integer configuration within the budget."""
        print('Generating all feasible solutions...')
        self.solutions = []

        # Any feasible count of one type cannot exceed its individual budget bound.
        for rtg in range(self.max_rtg + 1):
            for rmg in range(self.max_rmg + 1):
                for straddle in range(self.max_straddle + 1):
                    evaluation = self.evaluate_solution(rtg, rmg, straddle)
                    if evaluation['feasible']:
                        self.solutions.append(
                            {
                                'RTG': rtg,
                                'RMG': rmg,
                                'Straddle': straddle,
                                **evaluation,
                            }
                        )

        self.pareto_solutions = []
        print(f'Generated {len(self.solutions)} feasible solutions')
        return self.solutions

    def find_pareto_optimal(self) -> List[Dict]:
        """Return feasible solutions not dominated on the four modeled objectives."""
        if not self.solutions:
            self.generate_all_solutions()

        print('Finding Pareto-optimal solutions...')
        self.pareto_solutions = []
        objective_names = ('cost', 'productivity', 'flexibility', 'sustainability')

        for solution in self.solutions:
            is_dominated = False
            for other in self.solutions:
                if other is solution:
                    continue

                cost_no_worse = (
                    other['cost'] < solution['cost']
                    or math.isclose(
                        other['cost'], solution['cost'], rel_tol=1e-9, abs_tol=1e-12
                    )
                )
                maximize_no_worse = all(
                    other[name] > solution[name]
                    or math.isclose(
                        other[name], solution[name], rel_tol=1e-9, abs_tol=1e-12
                    )
                    for name in objective_names[1:]
                )
                strictly_better = (
                    other['cost'] < solution['cost']
                    and not math.isclose(
                        other['cost'], solution['cost'], rel_tol=1e-9, abs_tol=1e-12
                    )
                ) or any(
                    other[name] > solution[name]
                    and not math.isclose(
                        other[name], solution[name], rel_tol=1e-9, abs_tol=1e-12
                    )
                    for name in objective_names[1:]
                )

                if cost_no_worse and maximize_no_worse and strictly_better:
                    is_dominated = True
                    break

            if not is_dominated:
                self.pareto_solutions.append(solution)

        print(f'Found {len(self.pareto_solutions)} Pareto-optimal solutions')
        return self.pareto_solutions

    def weighted_sum_optimization(self, weights: Dict[str, float]) -> Optional[Dict]:
        """Select the feasible configuration with the highest weighted score."""
        required_weights = {'cost', 'productivity', 'flexibility', 'sustainability'}
        if set(weights) != required_weights:
            raise ValueError(f'Weights must have exactly these keys: {required_weights}.')
        if any(weight < 0 for weight in weights.values()):
            raise ValueError('Weights must be nonnegative.')
        if not math.isclose(sum(weights.values()), 1.0, rel_tol=1e-9, abs_tol=1e-12):
            raise ValueError('Weights must sum to 1.')

        if not self.solutions:
            self.generate_all_solutions()
        if not self.solutions:
            return None

        best_solution = None
        best_score = -math.inf

        for solution in self.solutions:
            cost_score = 1.0 - solution['cost'] / self.budget_limit
            productivity_score = solution['productivity'] / 1000.0
            flexibility_score = solution['flexibility']
            sustainability_score = solution['sustainability']

            score = (
                weights['cost'] * cost_score
                + weights['productivity'] * productivity_score
                + weights['flexibility'] * flexibility_score
                + weights['sustainability'] * sustainability_score
            )

            if score > best_score and not math.isclose(
                score, best_score, rel_tol=1e-9, abs_tol=1e-12
            ):
                best_score = score
                best_solution = solution.copy()
                best_solution['weighted_score'] = score

        return best_solution


print('Multi-objective optimization system defined.')

### Generate and analyze solutions

The first configuration is evaluated against all constraints before it is included in the feasible set. Pareto analysis then compares feasible configurations using the model's four objective directions. Finally, the listed weight scenarios illustrate how preference choices affect the selected configuration; they do not change the feasibility rules or the Pareto set.

In [ ]:
optimizer = MultiObjectiveEquipmentOptimizer(
    equipment_types=equipment_types,
    budget_limit=50.0,
    min_productivity=500,
)

all_solutions = optimizer.generate_all_solutions()
pareto_solutions = optimizer.find_pareto_optimal()

weight_scenarios = {
    'Balanced': {
        'cost': 0.25,
        'productivity': 0.25,
        'flexibility': 0.25,
        'sustainability': 0.25,
    },
    'Cost_Focused': {
        'cost': 0.5,
        'productivity': 0.2,
        'flexibility': 0.15,
        'sustainability': 0.15,
    },
    'Productivity_Focused': {
        'cost': 0.15,
        'productivity': 0.5,
        'flexibility': 0.2,
        'sustainability': 0.15,
    },
    'Flexibility_Focused': {
        'cost': 0.2,
        'productivity': 0.2,
        'flexibility': 0.4,
        'sustainability': 0.2,
    },
    'Sustainability_Focused': {
        'cost': 0.2,
        'productivity': 0.2,
        'flexibility': 0.15,
        'sustainability': 0.45,
    },
}

weighted_solutions = {
    scenario_name: optimizer.weighted_sum_optimization(weights)
    for scenario_name, weights in weight_scenarios.items()
}

print('\n=== SOLUTION ANALYSIS ===\n')
print(f'Total feasible solutions: {len(all_solutions)}')
print(f'Pareto-optimal solutions: {len(pareto_solutions)}')

print('\n=== WEIGHTED-SUM RESULTS ===\n')
for scenario, solution in weighted_solutions.items():
    print(f'{scenario}:')
    if solution is None:
        print('  No feasible configuration exists for this model instance.\n')
        continue
    print(
        f'  RTG: {solution["RTG"]}, RMG: {solution["RMG"]}, '
        f'Straddle: {solution["Straddle"]}'
    )
    print(
        f'  Capital cost: ${solution["cost"]:.1f}M, '
        f'Productivity: {solution["productivity"]:.0f} containers/hour'
    )
    print(f'  Weighted score: {solution["weighted_score"]:.3f}\n')

### Comprehensive visualization

The first panel shows cost against productivity, with Pareto configurations highlighted. The second shows the same two dimensions and encodes flexibility by color. The third compares the weighted-score components for the configuration selected in each scenario; productivity is divided by 1000 to match the score definition, while cost is represented by its budget-relative desirability. The final panel displays the equipment counts in the first ten Pareto configurations in enumeration order. These views help inspect trade-offs but do not replace the numerical feasibility and dominance tests.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    'Multi-Objective Equipment Selection', fontsize=16, fontweight='bold'
)

costs = [solution['cost'] for solution in all_solutions]
productivities = [solution['productivity'] for solution in all_solutions]
pareto_costs = [solution['cost'] for solution in pareto_solutions]
pareto_productivities = [
    solution['productivity'] for solution in pareto_solutions
]

ax1.scatter(costs, productivities, alpha=0.6, s=30, label='Feasible solutions')
ax1.scatter(
    pareto_costs,
    pareto_productivities,
    color='red',
    s=50,
    label='Pareto-optimal solutions',
)
ax1.set_xlabel('Capital cost (million USD)')
ax1.set_ylabel('Productivity (containers/hour)')
ax1.set_title('Capital Cost and Productivity')
ax1.grid(True, alpha=0.3)
ax1.legend()

flexibilities = [solution['flexibility'] for solution in pareto_solutions]
scatter = ax2.scatter(
    pareto_costs,
    pareto_productivities,
    c=flexibilities,
    cmap='viridis',
    s=55,
    alpha=0.8,
)
ax2.set_xlabel('Capital cost (million USD)')
ax2.set_ylabel('Productivity (containers/hour)')
ax2.set_title('Pareto Solutions: Color Shows Flexibility')
ax2.grid(True, alpha=0.3)
fig.colorbar(scatter, ax=ax2, label='Flexibility score')

scenarios = [
    scenario
    for scenario, solution in weighted_solutions.items()
    if solution is not None
]
scenario_solutions = [weighted_solutions[scenario] for scenario in scenarios]
cost_components = [
    1.0 - solution['cost'] / optimizer.budget_limit
    for solution in scenario_solutions
]
productivity_components = [
    solution['productivity'] / 1000.0 for solution in scenario_solutions
]
flexibility_components = [
    solution['flexibility'] for solution in scenario_solutions
]
sustainability_components = [
    solution['sustainability'] for solution in scenario_solutions
]

x = np.arange(len(scenarios))
width = 0.2
ax3.bar(x - 1.5 * width, cost_components, width, label='Cost desirability')
ax3.bar(x - 0.5 * width, productivity_components, width, label='Productivity / 1000')
ax3.bar(x + 0.5 * width, flexibility_components, width, label='Flexibility')
ax3.bar(x + 1.5 * width, sustainability_components, width, label='Sustainability')
ax3.set_xlabel('Weight scenario')
ax3.set_ylabel('Weighted-score component value')
ax3.set_title('Objective Components of Selected Solutions')
ax3.set_xticks(x)
ax3.set_xticklabels(
    [scenario.replace('_', ' ') for scenario in scenarios],
    rotation=35,
    ha='right',
)
ax3.legend(fontsize=8)
ax3.grid(True, axis='y', alpha=0.3)

composition_solutions = pareto_solutions[:10]
pareto_compositions = np.array(
    [
        [solution['RTG'], solution['RMG'], solution['Straddle']]
        for solution in composition_solutions
    ],
    dtype=int,
).reshape(-1, 3)
positions = np.arange(len(composition_solutions))
bottom = np.zeros(len(composition_solutions))
for column, label, color in (
    (0, 'RTG', 'steelblue'),
    (1, 'RMG', 'darkorange'),
    (2, 'Straddle', 'green'),
):
    ax4.bar(
        positions,
        pareto_compositions[:, column],
        bottom=bottom,
        label=label,
        color=color,
        alpha=0.75,
    )
    bottom += pareto_compositions[:, column]

ax4.set_xlabel('Pareto configuration index (enumeration order)')
ax4.set_ylabel('Number of units')
ax4.set_title('Equipment Mix in the First Pareto Configurations')
ax4.legend(title='Equipment type')
ax4.grid(True, axis='y', alpha=0.3)

fig.tight_layout()
plt.show()
print('Optimization visualizations completed.')

### Sensitivity analysis

This experiment repeats the same exhaustive enumeration and balanced weighted-sum selection for several budget limits while keeping the minimum productivity requirement fixed. A budget can be too small to admit any feasible configuration. Such a case is reported explicitly and represented as missing data in the plots; it is not treated as a zero-cost or zero-productivity solution.

The selected productivity in this analysis belongs to the balanced weighted-sum configuration, so it is not necessarily the maximum productivity attainable at that budget. Also, the cost desirability term in the weighted score depends on the budget being analyzed. Scores from different budget scenarios therefore should not be compared as though they used an identical cost scale.

In [ ]:
budget_scenarios = [30, 40, 50, 60, 70]
budget_analysis = []
balanced_weights = {
    'cost': 0.25,
    'productivity': 0.25,
    'flexibility': 0.25,
    'sustainability': 0.25,
}

print('=== BUDGET SENSITIVITY ANALYSIS ===\n')
for budget in budget_scenarios:
    optimizer_budget = MultiObjectiveEquipmentOptimizer(
        equipment_types=equipment_types,
        budget_limit=budget,
        min_productivity=500,
    )

    solutions = optimizer_budget.generate_all_solutions()
    pareto = optimizer_budget.find_pareto_optimal()
    best_solution = optimizer_budget.weighted_sum_optimization(balanced_weights)

    if best_solution is None:
        budget_analysis.append(
            {
                'budget': budget,
                'num_solutions': len(solutions),
                'num_pareto': len(pareto),
                'best_cost': np.nan,
                'best_productivity': np.nan,
                'best_units': np.nan,
            }
        )
        print(f'Budget ${budget}M: no feasible configuration.')
        print(f'  Feasible solutions: {len(solutions)}')
        print(f'  Pareto solutions: {len(pareto)}\n')
        continue

    budget_analysis.append(
        {
            'budget': budget,
            'num_solutions': len(solutions),
            'num_pareto': len(pareto),
            'best_cost': best_solution['cost'],
            'best_productivity': best_solution['productivity'],
            'best_units': best_solution['total_units'],
        }
    )
    print(f'Budget ${budget}M:')
    print(f'  Feasible solutions: {len(solutions)}')
    print(f'  Pareto solutions: {len(pareto)}')
    print(
        '  Balanced selection productivity: '
        f'{best_solution["productivity"]:.0f} containers/hour'
    )
    print(f'  Balanced selection capital cost: ${best_solution["cost"]:.1f}M\n')

budget_df = pd.DataFrame(budget_analysis)
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Budget Sensitivity Analysis', fontsize=16, fontweight='bold')

ax1.plot(
    budget_df['budget'],
    budget_df['num_solutions'],
    'bo-',
    linewidth=2,
    markersize=8,
)
ax1.set_xlabel('Capital budget (million USD)')
ax1.set_ylabel('Number of feasible configurations')
ax1.set_title('Feasible Set Size by Budget')
ax1.grid(True, alpha=0.3)

ax2.plot(
    budget_df['budget'],
    budget_df['num_pareto'],
    'ro-',
    linewidth=2,
    markersize=8,
)
ax2.set_xlabel('Capital budget (million USD)')
ax2.set_ylabel('Number of Pareto configurations')
ax2.set_title('Pareto Set Size by Budget')
ax2.grid(True, alpha=0.3)

ax3.plot(
    budget_df['budget'],
    budget_df['best_productivity'],
    'go-',
    linewidth=2,
    markersize=8,
)
ax3.set_xlabel('Capital budget (million USD)')
ax3.set_ylabel('Productivity (containers/hour)')
ax3.set_title('Productivity of the Balanced Selection')
ax3.grid(True, alpha=0.3)

cost_efficiency = budget_df['best_productivity'] / budget_df['best_cost']
ax4.plot(
    budget_df['budget'],
    cost_efficiency,
    'mo-',
    linewidth=2,
    markersize=8,
)
ax4.set_xlabel('Capital budget (million USD)')
ax4.set_ylabel('Containers/hour per million USD')
ax4.set_title('Productivity-to-Capital-Cost Ratio')
ax4.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()
print('Budget sensitivity analysis completed.')

### Results summary and interpretation

The summary below is generated from the computed feasible set and the balanced weight scenario. A weighted-sum selection is a preference-dependent recommendation within this model, not a universal operational recommendation. The Pareto set is useful when decision-makers want to inspect non-dominated trade-offs before choosing preferences. Conclusions about operating cost, maintenance, environmental compliance, or real terminal performance require those considerations to be modeled explicitly.

In [ ]:
print('=== MULTI-OBJECTIVE EQUIPMENT SELECTION: RESULTS ===\n')
print('Problem: Mediterranean Container Terminal equipment selection')
print('Method: Exhaustive enumeration, Pareto analysis, and weighted sums\n')

print('MODEL PARAMETERS:')
print(f'  Capital budget: ${optimizer.budget_limit:.1f}M')
print(
    f'  Minimum productivity: '
    f'{optimizer.min_productivity:.0f} containers/hour'
)
print('  Equipment types: RTG, RMG, and Straddle')
print('  Minimum equipment count: 3 units\n')

print('ENUMERATION RESULTS:')
print(f'  Feasible configurations: {len(all_solutions)}')
print(f'  Pareto-optimal configurations: {len(pareto_solutions)}')
print('  Search method: complete enumeration within budget-derived bounds\n')

balanced = weighted_solutions['Balanced']
if balanced is None:
    print('BALANCED WEIGHT SCENARIO:')
    print('  No feasible configuration exists for the specified model.')
else:
    print('BALANCED WEIGHT SCENARIO:')
    print(f'  RTG units: {balanced["RTG"]}')
    print(f'  RMG units: {balanced["RMG"]}')
    print(f'  Straddle units: {balanced["Straddle"]}')
    print(f'  Total equipment: {balanced["total_units"]} units\n')

    print('MODELED PERFORMANCE:')
    budget_utilization = balanced['cost'] / optimizer.budget_limit * 100
    print(
        f'  Capital cost: ${balanced["cost"]:.1f}M '
        f'(budget utilization: {budget_utilization:.1f}%)'
    )
    print(
        f'  Productivity: {balanced["productivity"]:.0f} containers/hour'
    )
    print(f'  Flexibility score: {balanced["flexibility"]:.3f}')
    print(f'  Sustainability proxy: {balanced["sustainability"]:.3f}')
    print(f'  Weighted score: {balanced["weighted_score"]:.3f}\n')

print('INTERPRETATION:')
print(
    '  The Pareto set identifies feasible configurations for which no '
    'other configuration is at least as good on every modeled objective '
    'and strictly better on one.'
)
print(
    '  The balanced selection reflects the supplied weights and score '
    'scales; it is not necessarily the preferred choice for every '
    'terminal.'
)
print(
    '  Operating cost, maintenance, footprint, and uncertainty are '
    'outside the current model.'
)